# Fuse multiple collections and evaluate a second-stage reranker

Implement BM25 candidate retrieval per collection, reciprocal rank fusion, and a transparent query–document scoring stage. Evaluate recall and ranking separately on a small labeled synthetic corpus. Optional cells show how hosted embeddings or a preapproved local cross-encoder can replace individual stages; the default lexical exercise does not demonstrate embedding or neural-model quality.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Write the evaluation set before tuning the ranking

Three collections hold fictional technical notes. Every document has a global identifier so a result returned by multiple retrievers can be deduplicated. Relevance judgments are authored labels for this small exercise, not benchmark ground truth. Keep the two final queries as a demonstration holdout; real evaluation needs substantially more independent judgments and uncertainty estimates.


In [ ]:
import math
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

records = [
    {
        "id": "thermal-1",
        "collection": "manuals",
        "text": "A heat pump moves thermal energy using a compressor and a refrigerant cycle.",
    },
    {
        "id": "thermal-2",
        "collection": "manuals",
        "text": "A blocked heat pump filter reduces airflow. Clean the filter before checking refrigerant pressure.",
    },
    {
        "id": "thermal-3",
        "collection": "research",
        "text": "Seasonal heat pump efficiency depends on outdoor temperature and the building heating demand.",
    },
    {
        "id": "storage-1",
        "collection": "manuals",
        "text": "A battery stores electrical energy. A charge controller prevents battery overcharging.",
    },
    {
        "id": "storage-2",
        "collection": "research",
        "text": "Battery capacity fades during repeated charge cycles and exposure to high temperature.",
    },
    {
        "id": "water-1",
        "collection": "operations",
        "text": "A water pump moves liquid through pipes. A clogged inlet filter can reduce water flow.",
    },
    {
        "id": "thermal-4",
        "collection": "operations",
        "text": "For low heat pump airflow, inspect the air filter and remove accumulated dust.",
    },
    {
        "id": "storage-3",
        "collection": "operations",
        "text": "Disconnect the battery charger before inspecting a damaged charge controller.",
    },
    {
        "id": "water-2",
        "collection": "research",
        "text": "Water flow meters estimate volume transported through a pipe per unit time.",
    },
]
by_id = {doc["id"]: doc for doc in records}
assert len(by_id) == len(records)
queries = [
    ("heat pump filter airflow", {"thermal-2": 2, "thermal-4": 2}, "development"),
    ("battery charge controller", {"storage-1": 2, "storage-3": 2}, "development"),
    ("heat pump seasonal efficiency", {"thermal-3": 2, "thermal-1": 1}, "holdout"),
    ("water flow pipes", {"water-1": 2, "water-2": 2}, "holdout"),
]


def tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())


collections = {
    name: [doc for doc in records if doc["collection"] == name]
    for name in sorted({doc["collection"] for doc in records})
}
print(pd.DataFrame(records)[["id", "collection", "text"]])

## 2. Retrieve per collection with BM25

For term t, use $\log(1+(N-df_t+0.5)/(df_t+0.5))$ as IDF and the standard BM25 length-normalized term-frequency factor with $k_1=1.2,b=0.75$. Each collection has its own statistics, so raw scores are not automatically comparable across collections. Fusion will use ranks instead. Zero-match documents are omitted.


In [ ]:
def bm25(query, documents, k=2, k1=1.2, b=0.75):
    if not documents or k <= 0:
        return []
    counts = [Counter(tokens(doc["text"])) for doc in documents]
    lengths = np.array([sum(c.values()) for c in counts])
    avg = max(float(lengths.mean()), 1)
    df = Counter(term for counts_i in counts for term in counts_i)
    scored = []
    for doc, terms, length in zip(documents, counts, lengths):
        score = 0.0
        for term in set(tokens(query)):
            frequency = terms[term]
            idf = math.log(1 + (len(documents) - df[term] + 0.5) / (df[term] + 0.5))
            denominator = frequency + k1 * (1 - b + b * length / avg)
            score += idf * frequency * (k1 + 1) / denominator
        if score > 0:
            scored.append((doc["id"], score))
    return sorted(scored, key=lambda pair: (-pair[1], pair[0]))[:k]


def rrf(rankings, constant=60):
    if constant <= 0:
        raise ValueError("RRF constant must be positive")
    total = defaultdict(float)
    for ranking in rankings:
        seen = set()
        for rank, (doc_id, _) in enumerate(ranking, 1):
            if doc_id not in seen:
                total[doc_id] += 1 / (constant + rank)
                seen.add(doc_id)
    return sorted(total.items(), key=lambda pair: (-pair[1], pair[0]))


query = queries[0][0]
rankings = [bm25(query, docs) for docs in collections.values()]
fused = rrf(rankings)
assert len(fused) == len({doc_id for doc_id, _ in fused})
assert rrf([[("x", 9), ("x", 8)]]) == [("x", 1 / 61)]
assert rrf([]) == [] and bm25("", records) == []
print("Per-collection candidates:", rankings)
print("Fused candidates:", fused)

## 3. Rescore query and passage jointly

The default scorer combines unique-token coverage and adjacent query-bigram matches. It reads the query and passage together but is **not a learned cross-encoder**. This transparent baseline exposes the stage contract and its failure modes: it misses synonyms, can be fooled by repetition or negation, and cannot recover a relevant item excluded during candidate retrieval. Reranking must preserve candidate identifiers and never invent new documents.


In [ ]:
def joint_lexical_score(query, document):
    q = tokens(query)
    d = tokens(document)
    if not q:
        return 0.0
    coverage = len(set(q) & set(d)) / len(set(q))
    q_pairs = set(zip(q, q[1:]))
    d_pairs = set(zip(d, d[1:]))
    phrase = len(q_pairs & d_pairs) / max(1, len(q_pairs))
    return coverage + 0.25 * phrase


def rerank(query, candidates):
    return sorted(
        [(doc_id, joint_lexical_score(query, by_id[doc_id]["text"])) for doc_id, _ in candidates],
        key=lambda pair: (-pair[1], pair[0]),
    )


rescored = rerank(query, fused)
assert {x[0] for x in rescored} == {x[0] for x in fused}
# Reranking cannot restore a document that was removed upstream.
missing = [x for x in fused if x[0] != "thermal-2"]
assert "thermal-2" not in {x[0] for x in rerank(query, missing)}
print(pd.DataFrame(rescored, columns=["id", "lexical_joint_score"]))

## 4. Measure retrieval coverage separately from ordering

Candidate recall is the fraction of judged relevant documents entering stage two. NDCG@3 uses gain $2^{rel}-1$ and logarithmic rank discount. Display every query, including cases where the simple reranker does not improve ranking. Labels and this small corpus were designed for exposition; do not report these scores as production retrieval performance.


In [ ]:
def ndcg(ranking, judgments, k=3):
    gains = [judgments.get(doc_id, 0) for doc_id, _ in ranking[:k]]

    def dcg(values):
        return sum((2**value - 1) / math.log2(rank + 2) for rank, value in enumerate(values))

    ideal = dcg(sorted(judgments.values(), reverse=True)[:k])
    return dcg(gains) / ideal if ideal else 0.0


rows = []
for query, judgments, split in queries:
    candidates = rrf([bm25(query, docs, k=2) for docs in collections.values()])
    present = {doc_id for doc_id, _ in candidates}
    recall = len(present & judgments.keys()) / len(judgments)
    rows.append(
        {
            "query": query,
            "split": split,
            "candidate_recall": recall,
            "RRF_NDCG@3": ndcg(candidates, judgments),
            "reranked_NDCG@3": ndcg(rerank(query, candidates), judgments),
        }
    )
metrics = pd.DataFrame(rows)
assert metrics["candidate_recall"].between(0, 1).all()
assert metrics[["RRF_NDCG@3", "reranked_NDCG@3"]].ge(0).all().all()
assert metrics[["RRF_NDCG@3", "reranked_NDCG@3"]].le(1 + 1e-12).all().all()
print(metrics.to_string(index=False))

## 5. Optional: replace stages with real models

Both switches remain false. For hosted embeddings, install the [audited optional retrieval profile](../../../_requirements/optional-retrieval.lock.txt), including `openai`, set `OPENAI_API_KEY` and `OPENAI_EMBEDDING_MODEL` in your process environment, and review the synthetic text that will leave the machine and the provider's costs. The client calls the official endpoint; no key is printed or saved. This example ranks one global pool to show the embedding stage; repeat it per collection before applying RRF in a deployment.

For a neural reranker, install a reviewed `sentence-transformers` profile and supply `RERANK_MODEL_DIR` containing a locally approved cross-encoder and tokenizer. Model acquisition, provenance, license, and weight-file review are separate tasks. `local_files_only=True` prevents a hidden model download; custom model code is disabled. These optional model paths have not been evaluated for retrieval quality here.

The hosted embedding response uses request-local indices: index 0 is the query and subsequent indices identify documents in submission order. Validate an exact index permutation and equally sized, finite, nonzero vectors before computing cosine similarity. The synthetic integrity checks below accept reversed response order and reject duplicate, missing, out-of-range, and boolean indices or malformed vectors. They test mapping safety, not embedding quality.


Provider-request and SDK parsing failures expose only an HTTP status or a generic error. Raw provider diagnostics may contain private values, so they are withheld from notebook outputs. Local schema and evidence-validation errors remain explicit.


In [ ]:
def validated_embedding_matrix(rows, expected_count):
    """Restore request order only after validating the complete response mapping."""
    if type(expected_count) is not int or expected_count < 1:
        raise ValueError("Expected count must be a positive integer")
    if not isinstance(rows, list) or len(rows) != expected_count:
        raise ValueError("Expected exactly one embedding per input")
    ordered = [None] * expected_count
    dimension = None
    for row in rows:
        if not isinstance(row, dict):
            raise ValueError("Each embedding must be an object")
        index = row.get("index")
        if type(index) is not int or not 0 <= index < expected_count:
            raise ValueError("Embedding index is outside the request")
        if ordered[index] is not None:
            raise ValueError("Duplicate embedding index")
        values = row.get("embedding")
        if not isinstance(values, list) or not values:
            raise ValueError("Embedding must be a nonempty numeric vector")
        if any(type(value) not in (int, float) for value in values):
            raise ValueError("Embedding values must be real numbers")
        vector = np.asarray(values, dtype=float)
        if not np.isfinite(vector).all():
            raise ValueError("Embedding values must be finite")
        if dimension is None:
            dimension = len(vector)
        if len(vector) != dimension:
            raise ValueError("Embedding dimensions must agree")
        ordered[index] = vector
    if any(vector is None for vector in ordered):
        raise ValueError("Embedding response is missing an input index")
    vectors = np.stack(ordered)
    with np.errstate(over="ignore", invalid="ignore", under="ignore"):
        norms = np.linalg.norm(vectors, axis=1)
    if not np.isfinite(norms).all() or np.any(norms == 0):
        raise ValueError("Embedding norms must be finite and nonzero")
    return vectors


# Synthetic response-contract tests run without the optional SDK or network.
embedding_fixture = [
    {"index": 0, "embedding": [1.0, 0.0]},
    {"index": 1, "embedding": [0.0, 1.0]},
    {"index": 2, "embedding": [1.0, 1.0]},
]
assert np.array_equal(
    validated_embedding_matrix(list(reversed(embedding_fixture)), 3),
    np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]),
)
invalid_embedding_responses = [
    [embedding_fixture[0], embedding_fixture[0], embedding_fixture[2]],
    embedding_fixture[:-1],
    [{"index": 3, "embedding": [1.0, 0.0]}, *embedding_fixture[1:]],
    [{"index": True, "embedding": [1.0, 0.0]}, *embedding_fixture[1:]],
]
for invalid_vector in (
    [],
    [1.0],
    [float("nan"), 1.0],
    [0.0, 0.0],
    [1e308, 1e308],
    [True, 1.0],
    [[1.0], [0.0]],
):
    invalid_embedding_responses.append(
        [{"index": 0, "embedding": invalid_vector}, *embedding_fixture[1:]]
    )
for invalid_response in invalid_embedding_responses:
    try:
        validated_embedding_matrix(invalid_response, 3)
    except ValueError:
        pass
    else:
        raise AssertionError("Malformed embedding response was accepted")
print("Embedding response mapping and vector-integrity checks passed.")

RUN_HOSTED_EMBEDDINGS = False
RUN_LOCAL_NEURAL_RERANKER = False
query = queries[0][0]
if RUN_HOSTED_EMBEDDINGS:
    import httpx2 as httpx  # OpenAI 3.x transport
    from openai import OpenAI

    if not os.environ.get("OPENAI_API_KEY") or not os.environ.get("OPENAI_EMBEDDING_MODEL"):
        raise ValueError("Set OPENAI_API_KEY and OPENAI_EMBEDDING_MODEL explicitly")
    with OpenAI(
        api_key=os.environ["OPENAI_API_KEY"],
        base_url="https://api.openai.com/v1",
        timeout=30.0,
        max_retries=0,
        http_client=httpx.Client(timeout=30.0, follow_redirects=False, trust_env=False),
    ) as client:
        try:
            response = client.embeddings.create(
                model=os.environ["OPENAI_EMBEDDING_MODEL"],
                input=[query] + [d["text"] for d in records],
            )
        except Exception as exc:
            status = getattr(exc, "status_code", None)
            if type(status) is int:
                raise RuntimeError(f"OpenAI embedding request failed with HTTP {status}") from None
            raise RuntimeError(
                "OpenAI embedding request failed; provider diagnostics were withheld"
            ) from None
    vectors = validated_embedding_matrix(
        [{"index": item.index, "embedding": item.embedding} for item in response.data],
        expected_count=len(records) + 1,
    )
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    normalized = vectors / norms
    scores = normalized[1:] @ normalized[0]
    print(sorted(zip(by_id, scores), key=lambda item: -item[1]))
if RUN_LOCAL_NEURAL_RERANKER:
    from sentence_transformers import CrossEncoder

    model_path = Path(os.environ["RERANK_MODEL_DIR"]).expanduser().resolve(strict=True)
    if not model_path.is_dir():
        raise ValueError("RERANK_MODEL_DIR must be a local model directory")
    model = CrossEncoder(str(model_path), local_files_only=True, trust_remote_code=False)
    candidates = rrf([bm25(query, docs) for docs in collections.values()])
    ids = [doc_id for doc_id, _ in candidates]
    scores = np.asarray(model.predict([(query, by_id[doc_id]["text"]) for doc_id in ids])).reshape(
        -1
    )
    assert len(scores) == len(ids) and np.isfinite(scores).all()
    print(sorted(zip(ids, scores), key=lambda item: -item[1]))
if not RUN_HOSTED_EMBEDDINGS and not RUN_LOCAL_NEURAL_RERANKER:
    print("Completed lexical evaluation. Optional embedding and neural stages were not run.")

## Limits, cleanup, and next steps

The default results validate algorithmic plumbing and metric calculations only. They do not prove neural cross-encoder, OpenAI, database, or production relevance quality. Scaling to Astra DB or another vector store additionally requires stable document IDs, consistent embedding dimensions, collection-specific access control, and labeled end-to-end evaluation. No collections are created or deleted here.

Default execution creates no files or remote resources. If opting in, remove credentials from the environment after use; manage any model directory under your model governance policy. Never paste keys into notebook cells or outputs.

Primary references: [Cormack, Clarke, and Buettcher, reciprocal rank fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/), [Sentence Transformers retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html), [OpenAI embeddings guide](https://developers.openai.com/api/docs/guides/embeddings).
